# September 4, 2026 corpus-paper audit

## tl;dr
The frozen archive contains 253 assessments, 237 comparable locked records,
187 religious-versus-skeptical comparisons, and 5,282 verified scored moves.
The mean substantive-position gap is 6.3422 points; the raw CON gap is 4.6962.
The 61.7% pooled untagged-loss majority reverses in the later process.
This notebook independently checks the exported counts and key arithmetic.
It does not re-judge transcripts or validate the AI's underlying interpretations.

## Context & Methods
Source revision: `76d006b37`. Reproduce source extraction with `analyze.py`.
The primary deliverables are seven PDFs; this is their inspectable audit companion.

### Key Assumptions
- Frozen, selected archive; no random-population claim.
- One observation per debate for paired gaps.
- Position classifications are explicit and contestable.
- Two scoring-process families must not be treated as a uniform tag detector.
- Resampling uncertainty does not include all source or judging errors.


## Data
### 1. Load the frozen exports

In [1]:
from pathlib import Path
import json, math, statistics
from collections import Counter
analysis_dir = Path.cwd()
debates = json.loads((analysis_dir / 'debates.json').read_text())
moves = json.loads((analysis_dir / 'moves.json').read_text())
results = json.loads((analysis_dir / 'results.json').read_text())
losses = json.loads((analysis_dir / 'losses.json').read_text())
print(f"{len(debates)} published assessments; {len(moves)} comparable scored moves")

253 published assessments; 5282 comparable scored moves


### 2. Validate identifiers, coverage, and join counts

In [2]:
assert len({row['id'] for row in debates}) == len(debates) == 253
assert len({row['youtube'] for row in debates}) == 253
assert len({(move['number'], move['move_id']) for move in moves}) == len(moves) == 5282
comparable = [row for row in debates if row['cohort'] != 'unlocked']
religious = [row for row in comparable if row['theist_side']]
assert len(comparable) == 237 and len(religious) == 187
assert sum(row['public_moves'] for row in debates) == 5492
print('Process counts:', dict(Counter(row['cohort'] for row in debates)))
print('Scoring checks:', results['checks'])

Process counts: {'earlier': 179, 'unlocked': 16, 'later': 58}
Scoring checks: {'move_scores': 5282, 'section_scores': 2558, 'overall_scores': 474, 'public_move_scores': 5282, 'final_ledger_hashes': 58}


## Results
### 3. Recompute the main means directly from exported debate rows

In [3]:
position_sum = sum(row[row['non_side']] - row[row['theist_side']] for row in religious)
role_sum = sum(row['con'] - row['pro'] for row in comparable)
assert position_sum == 1186 and role_sum == 1113
assert math.isclose(position_sum / 187, results['p1']['gap']['mean'])
assert math.isclose(role_sum / 237, results['p4']['raw']['mean'])
print(f"Position gap: {position_sum} / 187 = {position_sum / 187:.6f} points")
print(f"Role gap: {role_sum} / 237 = {role_sum / 237:.6f} points")
print('Topic counts:', {t['topic']: t['n'] for t in results['p2']['topics']})
assert sum(t['n'] for t in results['p2']['topics']) == 187

Position gap: 1186 / 187 = 6.342246 points
Role gap: 1113 / 237 = 4.696203 points
Topic counts: {'Religion, culture & meaning': 22, 'Scripture, revelation & doctrine': 22, 'Mind, reason & logic': 20, 'Evil, suffering & hiddenness': 20, 'Morality & moral foundations': 21, 'Cosmology, science & design': 24, 'General theism & naturalism': 41, 'Resurrection & historical evidence': 17}


### 4. Reconcile the weighted score gap

In [4]:
contributions = results['p1']['contributions']
reconstructed = sum(contributions.values()) + results['p1']['rounding'] + results['p1']['adjustment']
assert math.isclose(reconstructed, position_sum / 187)
for dimension, value in contributions.items():
    print(f"{dimension}: {value:.4f} overall points")
print(f"With rounding and adjustment: {reconstructed:.6f}")

logicalCoherence: 1.7774 overall points
evidenceWarrant: 1.6960 overall points
responsiveness: 1.2961 overall points
relevanceBurden: 0.2894 overall points
precisionClarity: 0.5126 overall points
calibrationCharity: 0.8015 overall points
With rounding and adjustment: 6.342246


### 5. Test the pooled fallacy claim against each process family

In [5]:
assert len(losses) == 243
assert sum(row['lower_no_fallacy'] for row in losses) == 150
for family in ['earlier', 'later', 'unlocked']:
    selected = [row for row in losses if row['cohort'] == family]
    untagged = sum(row['lower_no_fallacy'] for row in selected)
    print(f"{family}: {untagged}/{len(selected)} = {100*untagged/len(selected):.1f}%")
print(f"Combined: 150/243 = {100*150/243:.1f}%")

earlier: 139/172 = 80.8%
later: 8/55 = 14.5%
unlocked: 3/16 = 18.8%
Combined: 150/243 = 61.7%


### 6. Check rank-field size and the small neighboring gaps

In [6]:
ranking = results['p7']['ranking']
assert len(ranking) == 50
assert sum(row['n'] for row in ranking) == 334
adjacent = [a['mean'] - b['mean'] for a, b in zip(ranking, ranking[1:])]
print(f"50 ranked speakers; median neighboring gap {statistics.median(adjacent):.4f}")
print('Median empirical rank width:', results['p7']['median_empirical_rank_width'])
print('Median model rank width:', results['p7']['median_model_rank_width'])

50 ranked speakers; median neighboring gap 0.1714
Median empirical rank width: 12.0
Median model rank width: 19.0


## Takeaways
- The key totals and headline gaps reconcile.
- The position gap is a performance-score contrast, not a probability that a worldview is true.
- The original slogan-risk rule covers 146 debates; the separate final-score check covers 187.
- The archive-wide untagged-loss majority does not describe the later process.
- Rank averages are informative, but neighboring positions are far more precise-looking than the evidence warrants.

For full resampling, the 51-speaker scale bridge, and rank-model calculations, run `analyze.py`.
For PDF checks and rendered-page contacts, run `verify_papers.py`.
The seven manuscripts put the limitations next to the relevant findings.
